# molprop-ensemble — Notebook Utama

Pipeline ensemble **ECFP+RF · ChemBERTa · D-MPNN** + SMILES-Enumeration TTA (BBBP, BACE, ClinTox).

### Cara pakai — cukup **Run All**
Tidak ada pilihan/centang. Notebook **selalu full run** (3 dataset × 5 seed) dan **otomatis**:
- mengecek apakah ada artefak tersimpan yang **valid** (versi cocok) → dipakai ulang (resume),
- membersihkan artefak **basi** (kalau kode/algoritma berubah) → sekali, otomatis.

Jadi **tidak perlu reset manual**. Kalau sesi terputus, cukup Run All lagi — training lanjut dari checkpoint terakhir.

**Prasyarat panel kanan:** Accelerator = **GPU T4 x2**, Internet = **On**, dan Secret **`GH_TOKEN`** bila repo privat (lihat `KAGGLE.md`).

> ⏱️ Full run memakan waktu lama (bisa berjam-jam). Sesi Kaggle ~9–12 jam; resume otomatis aktif.

## 1. Setup — clone, install, cek environment

In [ ]:
# 1a. Clone repo dari GitHub (pakai GH_TOKEN bila repo privat)
REPO_OWNER, REPO_NAME, REPO_DIR = "belvahector-ship-it", "pharm_", "pharm_"
import os, subprocess
if not os.path.exists(REPO_DIR):
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    url = (f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if token
           else f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git")
    print("clone via GH_TOKEN..." if token else "GH_TOKEN tak ada -> coba clone publik.")
    r = subprocess.run(["git", "clone", url, REPO_DIR], capture_output=True, text=True,
                       timeout=120, stdin=subprocess.DEVNULL)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("Clone gagal — repo privat tanpa GH_TOKEN valid? Lihat KAGGLE.md.")
else:
    print(f"{REPO_DIR} sudah ada, skip clone.")
%cd {REPO_DIR}
!git pull --quiet && git log --oneline -1

In [ ]:
# 1b. Install dependency (chemprop 2.x; deepchem tidak diperlukan)
!pip install -q rdkit "chemprop>=2.1,<3.0" transformers
print("install selesai")

In [ ]:
# 1c. Cek environment (WAJIB 2 GPU) + versi paket
import torch
print("torch:", torch.__version__, "| GPU:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  GPU", i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, "Set Accelerator = GPU T4 x2 di panel kanan, lalu restart."
import rdkit, sklearn, transformers, chemprop
print("rdkit", rdkit.__version__, "| sklearn", sklearn.__version__,
      "| transformers", transformers.__version__, "| chemprop", chemprop.__version__)

## 2. Data & scaffold split
`01_prepare_data` otomatis mengecek versi artefak: valid → dipakai ulang, basi → dibersihkan sekali. Diagnostik memastikan split sah (**overlap SCAFFOLD = 0**).

In [ ]:
# 2a. Fase 1 — scaffold split (auto reuse/refresh berbasis versi)
!python scripts/01_prepare_data.py

In [ ]:
# 2b. Diagnostik mutu split — WAJIB "overlap SCAFFOLD = 0" & "bocor->train = 0.0%"
!python scripts/00_diagnose_split.py

## 3. Jalankan pipeline — FULL (3 dataset × 5 seed)
Satu alur: **training (Fase 4) → TTA (5) → fusion (6) → evaluasi (7)**. ChemBERTa→GPU0, D-MPNN→GPU1, RF→CPU (paralel). Yang sudah ada di-skip/resume otomatis, jadi aman dijalankan ulang bila sesi terputus.

In [ ]:
import subprocess, time

# Fase 4 — training SEMUA dataset x SEMUA seed, paralel: ChemBERTa->GPU0, D-MPNN->GPU1, RF->CPU.
# Log ketiga proses mengalir langsung ke sel ini (interleaved). Dipantau via polling supaya
# progres terlihat & GAGAL cepat ketahuan (tidak menunggu proses lain sia-sia).
#
# PENTING: kalau sel ini di-interrupt (tombol Stop / Cancel Run / KeyboardInterrupt) SAAT
# training masih jalan, proses child (chemberta/dmpnn/rf) BISA JADI TETAP JALAN di background
# sbg proses "yatim" memakai GPU/CPU diam-diam -- notebook kelihatan berhenti padahal belum.
# try/except/finally di bawah memastikan SEMUA proses child di-terminate begitu sel berhenti
# dgn cara apa pun (interrupt, error, atau selesai normal) -- state selalu bersih & terprediksi.
procs = {m: subprocess.Popen(["python", "scripts/02_train_baselines.py", "--model", m])
         for m in ["chemberta", "dmpnn", "rf"]}
print("Fase 4 dimulai (paralel). Menunggu... (ChemBERTa/D-MPNN bisa lama)")
done, t0 = {}, time.time()
try:
    while len(done) < len(procs):
        for m, p in procs.items():
            if m in done:
                continue
            rc = p.poll()
            if rc is not None:
                done[m] = rc
                print(f"[Fase4 {m}] {'OK' if rc == 0 else 'GAGAL'} (rc={rc}) @ {int(time.time()-t0)}s")
                if rc != 0:
                    raise RuntimeError(f"training {m} gagal (rc={rc}) — cek log/outputs/logs/.")
        if len(done) < len(procs):
            time.sleep(10)

    # Fase 5-7 (semua dataset x seed), sekuensial
    for script in ["04_run_tta.py", "03_run_fusion.py", "05_evaluate.py"]:
        print(f"\n>>> {script}")
        rc = subprocess.run(["python", f"scripts/{script}"]).returncode
        assert rc == 0, f"{script} gagal (rc={rc})"
    print("\nPIPELINE FULL SELESAI.")
except (KeyboardInterrupt, Exception):
    print("\n[cleanup] Sel dihentikan/gagal -> menghentikan SEMUA proses training background...")
    for name, p in procs.items():
        if p.poll() is None:  # masih jalan
            p.terminate()
            print(f"  [cleanup] {name} (pid={p.pid}) dihentikan.")
    raise
finally:
    # Pastikan tak ada proses tersisa meski terminate() di atas gagal/lambat.
    for name, p in procs.items():
        if p.poll() is None:
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                p.kill()
                print(f"  [cleanup] {name} (pid={p.pid}) dipaksa kill (tak merespons terminate).")

## 4. Hasil
Tabel final: `roc_auc_mean±std` (5 seed), **bootstrap 95% CI**, **p-value** vs baseline post-hoc, **Cohen's d**. Dengan full run 5 seed, kolom p-value & std kini bermakna.

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None, "display.width", 200)
pd.read_csv("outputs/results/final_table.csv")

## 4a. Tuning v1 — Adaptive TTA gating (hasil TERPISAH, tidak overwrite tes 1)
Prioritas 1 dari `peta-konsep-dan-tuning.md`: matikan TTA otomatis untuk dataset sangat
imbalanced (ClinTox) + laporkan stacking. Murni post-processing prediksi (tanpa retraining).
Hasil ditulis ke `outputs/results/tuned_v1/` — tes 1 (`final_table.csv`) tetap utuh.

In [ ]:
!python scripts/07_evaluate_tuned.py
import pandas as pd
print("\n=== Tabel TUNED ===")
display(pd.read_csv("outputs/results/tuned_v1/final_table_tuned.csv"))
print("=== Komparasi tes 1 vs tuned (yang berubah) ===")
cmp = pd.read_csv("outputs/results/tuned_v1/comparison_tes1_vs_tuned.csv")
display(cmp[cmp["delta"].abs() > 1e-9] if "delta" in cmp else cmp)

## 4c. Tuning v2 — Cari konfigurasi TERBAIK (subset selektif + kalibrasi, hasil TERPISAH)
Prioritas 3+4: coba semua subset anggota × strategi fusi × kalibrasi Platt, dipilih via
**5-fold CV di dalam val set** (bukan in-sample — mencegah stacking curang menang seleksi
krn overfit ke val). Hasil ke `outputs/results/tuned_v2_best/` (tes 1 & tuned_v1 tetap utuh).

In [ ]:
!python scripts/08_evaluate_best.py
import pandas as pd
print("\n=== Metode TERBAIK per dataset ===")
display(pd.read_csv("outputs/results/tuned_v2_best/final_table_best.csv"))
print("=== Perbandingan 3 tahap: tes1 vs tuned_v1 vs tuned_v2_best ===")
display(pd.read_csv("outputs/results/tuned_v2_best/comparison_all_stages.csv"))

## 4b. Simpan hasil ke GitHub (permanen)
**Kenapa perlu:** `/kaggle/working` bersifat *ephemeral* — begitu sesi berakhir tanpa langkah
eksplisit, `outputs/results/final_table.csv` dkk **hilang** (hanya ada di zip yang harus diunduh
manual). Sel ini otomatis **commit + push** `outputs/results/` (file kecil: csv/json/txt saja,
BUKAN checkpoint/prediksi besar) ke repo GitHub, jadi hasil tersimpan permanen & bisa
dibandingkan antar run lewat riwayat commit.

In [ ]:
import subprocess, datetime

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode

# Identitas git (Kaggle biasanya belum diset -> tanpa ini commit bisa gagal)
subprocess.run(["git", "config", "user.email", "kaggle-runner@example.com"], capture_output=True)
subprocess.run(["git", "config", "user.name", "Kaggle Runner"], capture_output=True)

_git("add", "outputs/results")
staged = subprocess.run(["git", "diff", "--cached", "--quiet"])
if staged.returncode == 0:
    print("Tidak ada perubahan baru di outputs/results/ -> tidak ada yang di-commit.")
else:
    code_sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip()
    msg = f"Hasil run Kaggle {datetime.datetime.now():%Y-%m-%d %H:%M} (kode @ {code_sha})"
    if _git("commit", "-m", msg) == 0:
        # Push pakai GH_TOKEN yang sama dgn clone (repo privat) bila tersedia.
        token = None
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GH_TOKEN")
        except Exception:
            pass
        push_target = (f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
                       if token else "origin")
        rc = _git("push", push_target, "HEAD:main")
        if rc == 0:
            print("\n✓ outputs/results/ ter-push ke GitHub -> aman meski sesi Kaggle berakhir.")
        else:
            print("\n⚠ Push gagal (lihat pesan di atas). Commit tetap tersimpan LOKAL di sesi "
                  "ini -> unduh manual via zip di bawah sbg cadangan.")

## 5. Simpan hasil (cadangan)
`outputs/` (selain `results/`) di-gitignore — checkpoint/prediksi besar tak ikut GitHub.
Zip sbg cadangan tambahan di luar `outputs/results/` yang sudah di-push di atas.

In [ ]:
!pip freeze > outputs/results/pip_freeze.txt
!cd outputs && zip -rq /kaggle/working/hasil_outputs.zip .
print("tersimpan: /kaggle/working/hasil_outputs.zip  &  outputs/results/{environment,pip_freeze}.txt")

In [ ]:
# 5b. Download SEMUA hasil ke komputer lokal — satu file zip, satu klik (cadangan bila push gagal)
import os, shutil
from IPython.display import FileLink, display

zip_name = "hasil_outputs"
if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")
shutil.make_archive(zip_name, "zip", "outputs")

size_mb = os.path.getsize(f"{zip_name}.zip") / (1024 * 1024)
print(f"Zip dibuat: {zip_name}.zip ({size_mb:.1f} MB)\n")
print("KLIK link biru di bawah ini untuk unduh ke komputer lokal Anda:\n")
display(FileLink(f"{zip_name}.zip"))

print("\nKalau link di atas TIDAK bisa diklik (kadang terjadi di Kaggle),")
print("unduh manual lewat panel kanan -> Output -> cari 'hasil_outputs.zip',")
print("atau klik 'Save Version' lalu unduh dari tab Output notebook version tsb.")